## Test integracyjny i logika SCD2

Sprawdzamy, czy ładowanie przyrostowe zostało poprawnie zaimplementowane, czyli czy po dodaniu nowych danych w warstwie bronze, jesteśmy w stanie przejść cały pipeline aż do warstwy gold. 

### Test: Dodanie nowych danych do tabel *bronze_airlines* i *bronze_airports*

In [0]:
# %python
# spark.sql(f"DELETE FROM flights_bronze.airlines WHERE iata_code = '{TEST_AIR_CODE}'")
# spark.sql(f"DELETE FROM flights_bronze.airports WHERE iata_code = '{TEST_APT_CODE}'")

DataFrame[num_affected_rows: bigint]

In [0]:
%python

# nowe dane testowe
TEST_AIR_CODE = "TEST_AIR"
TEST_AIR_NAME = "Test Airline Special"
TEST_APT_CODE = "TST"
TEST_APT_NAME = "Test International Airport"
TEST_YEAR = 2035

# dane dla istniejących rekordów
UPDATE_AIR_CODE = "VX"
UPDATE_AIR_NAME_OLD = "Virgin America"
UPDATE_AIR_NAME_NEW = "Virgin America REBRANDED"

UPDATE_APT_CODE = "YUM"
UPDATE_APT_NAME_OLD = "Yuma International Airport"
UPDATE_APT_NAME_NEW = "Yuma Airport CHANGED"     


In [0]:
%python
from pyspark.sql.functions import lit, col

# dodanie nowej linii lotniczej
df_airline_test = spark.createDataFrame(
    [(TEST_AIR_CODE, TEST_AIR_NAME),
     (UPDATE_AIR_CODE, UPDATE_AIR_NAME_NEW)
    ], 
    ["iata_code", "airline"]
)

# dodanie nowego lotniska
df_airport_test = spark.createDataFrame(
    [(TEST_APT_CODE, TEST_APT_NAME, "Test City", "TS", "USA", 0.0, 0.0),
     (UPDATE_APT_CODE, UPDATE_APT_NAME_NEW, "Yuma", "AZ", "USA", 32.65658, -114.60597)
     ], 
    ["iata_code", "airport", "city", "state", "country", "latitude", "longitude"]
)

# dodanie danych do warstwy bronze
df_airline_test.write.format("delta").mode("append").saveAsTable("flights_bronze.airlines")
df_airport_test.write.format("delta").mode("append").saveAsTable("flights_bronze.airports")

print("SUCCESS: Data inserted into bronze layer.")

# próba przetworzenia pipeline'u
try:
    # silver_airlines i silver_airports
    dbutils.notebook.run("../2_silver/2-1_silver_airlines", 600)
    dbutils.notebook.run("../2_silver/2-2_silver_airports", 600)
    
    print("SUCCESS: Notebooks executed correctly.\n")
    
except Exception as e:
    print(f"ERROR: {e}")

# weryfikacja
silver_airline_check = spark.table("flights_silver.dim_airlines") #\    .filter(col("iata_code") == TEST_AIR_CODE)

silver_airport_check = spark.table("flights_silver.dim_airports")# \    .filter(col("iata_code") == TEST_APT_CODE)

# wyświetlamy wyniki
display(spark.table("flights_silver.dim_airportss")
        .orderBy(col("iata_code").desc())
        .limit(5)
)

display(spark.table("flights_silver.dim_airlines")
        .orderBy(col("iata_code").desc())
        .limit(5)
)

# Prosta asercja, żebyś widziała w logach czy jest OK
if silver_airline_check.count() == 1 and silver_airport_check.count() == 1:
    print("SUCCESS: New airport and airline added correctly to the silver layer.")
else:
    print("FAIL: No new records in silver layer.")



SUCCESS: Data inserted into bronze layer.
SUCCESS: Notebooks executed correctly.



airport_key,iata_code,airport_name,city,state,country,latitude,longitude,is_current,start_date,end_date
YUM,YUM,Yuma Airport CHANGED,Yuma,AZ,USA,32.65658,-114.60597,false,1900-01-01,2025-12-10
YUM,YUM,Yuma International Airport,Yuma,AZ,USA,32.65658,-114.60597,false,1900-01-01,2025-12-10
YAK,YAK,Yakutat Airport,Yakutat,AK,USA,59.50336,-139.66023,true,1900-01-01,null
XNA,XNA,Northwest Arkansas Regional Airport,Fayetteville/Springdale/Rogers,AR,USA,36.28187,-94.30681,true,1900-01-01,null
WYS,WYS,Westerly State Airport,West Yellowstone,MT,USA,44.6884,-111.11764,true,1900-01-01,null


airline_key,iata_code,airline_name,is_current,start_date,end_date
WN,WN,Southwest Airlines Co.,true,1900-01-01,null
VX,VX,Virgin America REBRANDED,false,1900-01-01,2025-12-10
VX,VX,Virgin America,false,1900-01-01,2025-12-10
US,US,US Airways Inc.,true,1900-01-01,null
UA,UA,United Air Lines Inc.,true,1900-01-01,null


FAIL: No new records in silver layer.


### Test: Dodanie nowego wiersza do tabeli *bronze_flights*

In [0]:
%python
from pyspark.sql.functions import col, lit

df_bronze_schema = spark.table("flights_bronze.flights").limit(1)

# sztuczny wiersz
df_test_row = df_bronze_schema \
    .withColumn("year", lit(2025)) \
    .withColumn("month", lit(1)) \
    .withColumn("day", lit(1)) \
    .withColumn("day_of_week", lit(3)) \
    .withColumn("airline", lit("AA")) \
    .withColumn("flight_number", lit(798)) \
    .withColumn("tail_number", lit("TEST_999")) \
    .withColumn("origin_airport", lit("JFK")) \
    .withColumn("destination_airport", lit("LAX")) \
    .withColumn("scheduled_departure", lit("1200")) \
    .withColumn("departure_time", lit("1200")) \
    .withColumn("departure_delay", lit(0)) \
    .withColumn("taxi_out", lit(15)) \
    .withColumn("wheels_off", lit("1215")) \
    .withColumn("scheduled_time", lit(300)) \
    .withColumn("elapsed_time", lit(300)) \
    .withColumn("air_time", lit(270)) \
    .withColumn("distance", lit(2475)) \
    .withColumn("wheels_on", lit("1645")) \
    .withColumn("taxi_in", lit(15)) \
    .withColumn("scheduled_arrival", lit("1700")) \
    .withColumn("arrival_time", lit("1700")) \
    .withColumn("arrival_delay", lit(0)) \
    .withColumn("diverted", lit(0)) \
    .withColumn("cancelled", lit(0)) \
    .withColumn("cancellation_reason", lit(None).cast("string")) \
    .withColumn("air_system_delay", lit(0)) \
    .withColumn("security_delay", lit(0)) \
    .withColumn("airline_delay", lit(0)) \
    .withColumn("late_aircraft_delay", lit(0)) \
    .withColumn("weather_delay", lit(0))

# zapis do warstwy bronze
df_test_row.write.format("delta").mode("append").saveAsTable("flights_bronze.flights")

# próba przejścia przez kolejne notebooki dotyczące flights
try:
    # silver_flights
    dbutils.notebook.run("../2_silver/2-3_silver_flights", 600)
    # gold_fact_table
    dbutils.notebook.run("../3_gold/3-1_gold_fact_table", 600)
    print("Notebooks executed correctly.\n")
except Exception as e:
    print(f"Error: {e}")

# weryfikacja w gold
df_check = spark.table("flights_gold.fact_flights").filter("year == 2025 AND flight_number == '798'")

if df_check.count() == 1:
    print("SUCCESS: Flight made it to the gold layer.")
    display(df_check)
else:
    print("FAILURE: Flight not found in the gold layer.")

Error: An error occurred while calling o128.run.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Workload failed, see run output for details
	at com.databricks.workflow.WorkflowDriver.run(WorkflowDriver.scala:98)
	at com.databricks.dbutils_v1.impl.NotebookUtilsImpl.run(NotebookUtilsImpl.scala:130)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:397)
	at py4j.Gateway.invoke(Gateway.java:306)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:

In [0]:
# usunięcie wiersza
spark.sql("DELETE FROM flights_bronze.flights WHERE year = 2025")
spark.sql("DELETE FROM flights_silver.flights WHERE year = 2025")
spark.sql("DELETE FROM flights_gold.fact_flights WHERE year = 2025")

DataFrame[num_affected_rows: bigint]

### Sprzątanie po testach

In [0]:
spark.sql(f"DELETE FROM flights_bronze.airlines WHERE iata_code = '{TEST_AIR_CODE}'")
spark.sql(f"DELETE FROM flights_bronze.airports WHERE iata_code = '{TEST_APT_CODE}'")
spark.sql(f"DELETE FROM flights_silver.dim_airlines WHERE iata_code = '{TEST_AIR_CODE}'")
spark.sql(f"DELETE FROM flights_silver.dim_airports WHERE iata_code = '{TEST_APT_CODE}'")

print(f"   Przywracanie nazwy dla {UPDATE_AIR_CODE} -> {UPDATE_AIR_NAME_OLD}")
spark.sql(f"""
    UPDATE flights_silver.dim_airlines 
    SET airline = '{UPDATE_AIR_NAME_OLD}' 
    WHERE iata_code = '{UPDATE_AIR_CODE}'
""")

print(f"   Przywracanie nazwy dla {UPDATE_APT_CODE} -> {UPDATE_APT_NAME_OLD}")
spark.sql(f"""
    UPDATE flights_silver.dim_airports 
    SET airport = '{UPDATE_APT_NAME_OLD}' 
    WHERE iata_code = '{UPDATE_APT_CODE}'
""")

spark.sql(f"DELETE FROM flights_bronze.airlines WHERE airline = '{UPDATE_AIR_NAME_NEW}'")
spark.sql(f"DELETE FROM flights_bronze.airports WHERE airport = '{UPDATE_APT_NAME_NEW}'")

spark.sql("DELETE FROM flights_bronze.flights WHERE year = 2025")
spark.sql("DELETE FROM flights_silver.flights WHERE year = 2025")
spark.sql("DELETE FROM flights_gold.fact_flights WHERE year = 2025")

   Przywracanie nazwy dla VX -> Virgin America


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5509552528510240>, line 12
      7 # 2. PRZYWRACAMY oryginalne nazwy dla "starych" rekordów (UPDATES)
      8 # Ważne: Musimy to zrobić w Silver, żeby baza wróciła do stanu sprzed testu.
      9 # W Bronze teoretycznie też można usunąć te wiersze appendowane, ale update w Silver jest kluczowy.
     11 print(f"   Przywracanie nazwy dla {UPDATE_AIR_CODE} -> {UPDATE_AIR_NAME_OLD}")
---> 12 spark.sql(f"""
     13     UPDATE flights_silver.dim_airlines 
     14     SET airline = '{UPDATE_AIR_NAME_OLD}' 
     15     WHERE iata_code = '{UPDATE_AIR_CODE}'
     16 """)
     18 print(f"   Przywracanie nazwy dla {UPDATE_APT_CODE} -> {UPDATE_APT_NAME_OLD}")
     19 spark.sql(f"""
     20     UPDATE flights_silver.dim_airports 
     21     SET airport = '{UPDATE_APT_NAME_OLD}' 
     22     WHERE iata_code = '{UPDATE_APT_CODE}'
     23 